# Point Predictors
### Notebook initialisation

In [ ]:
%load_ext autoreload
%autoreload 2
import os
os.environ['PYTHONOPTIMIZE'] = '1'


print(__debug__)

import logging
logging.basicConfig(level=logging.INFO)


import torch, random, os, cv2
import numpy as np

seed = 1

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
cv2.setRNGSeed(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
os.environ["PYTHONHASHSEED"] = str(seed)

import seaborn as sns
sns.set_theme(style="whitegrid")

### Dataset loading:

In [ ]:
robot_data_folder_location = "../example_datasets/example_small_aruco1"
vrs_file_location = "../example_datasets/small_aruco1_sitting_20fps.vrs"

from shared import CompleteRobotScan
from headset_localization import HeadsetRecording, bind_headset_recording_to_scan
from headset_localization import Scanned3dEnvironment, visualize_robot_camera_environment_combo, XYZImageGenerationConfig, ICPAlignmentConfig

robot_data = CompleteRobotScan.from_folder(robot_data_folder_location)
robot_env = Scanned3dEnvironment.from_gathered_robot_data(
        robot_data = robot_data,
        number_of_sampled_datapoints=10,
        sample_datapoints_based_on_aruco_corectness = False,
        only_sample_robot_datapoints_w_marker_estimates = True,
        markers_use_advanced_removal=True,
        est3d_xyz_image_gen_config = XYZImageGenerationConfig(iforest_contamination=0.05, use_depth_images_if_provided=True),
        est3d_xyz_icp_config=ICPAlignmentConfig(do_alginment=True)
)

labeled_headset_data = bind_headset_recording_to_scan(
        headset_data = HeadsetRecording.from_vrs_file(vrs_file_location),
        robot_data = robot_data
)

visualize_loaded_data = False

if visualize_loaded_data:
    visualize_robot_camera_environment_combo(robot_env=robot_env, headset_data=labeled_headset_data, vis_headset_camera_wireframes = True, headset_frame_size=0.02)

## Hyperparameters
Untuned baseline:

In [ ]:
from headset_localization import PnPLocalizer, ExtractAndMatchWrapperConfig, PredictionOnDataset, FastGrippingError

default_point_predictor = PnPLocalizer(
        cam2_intrinsic_mtx=labeled_headset_data.intrinsic_cam_mtx,
        cam1_bgr_images=robot_env.robot_bgr_images,
        cam1_xyz_images=robot_env.robot_xyz_images,
        extract_and_match_wrapper_config=ExtractAndMatchWrapperConfig()
)

PredictionOnDataset(
    predictor = default_point_predictor,
    headset_data = labeled_headset_data, 
    gripping_error=FastGrippingError(points=robot_env.robot_xyz_images, intrinsics=labeled_headset_data.intrinsic_cam_mtx, visualize=False)
).print_summary()

## Extract and Match Options

The following `ExtractAndMatch` options are available: 

In [ ]:
from headset_localization import GradableLocalizer, NPredictors1DatasetGrader
from headset_localization import ExtractAndLightGlue, ExtractAndMatchLoMa, ExtractAndMatchEffLoFTR


lightglue_variants = [
        GradableLocalizer(
            creator=PnPLocalizer.get_creation_function(
                cam2_intrinsic_mtx=labeled_headset_data.intrinsic_cam_mtx,
                extract_and_match_wrapper_config=ExtractAndMatchWrapperConfig(
                    extract_and_match=ExtractAndLightGlue(extractor=extractor)
                ),
            ),
            category = "LG",
            name=f"{extractor}"
        )
        for extractor in ['SuperPoint', 'DISK', 'SIFT', 'ALIKED', 'DogHardNet']
]

loma_variants = [
    GradableLocalizer(
        creator=PnPLocalizer.get_creation_function(
                cam2_intrinsic_mtx=labeled_headset_data.intrinsic_cam_mtx,
                extract_and_match_wrapper_config=ExtractAndMatchWrapperConfig(
                    extract_and_match=ExtractAndMatchLoMa(loma_variant=variant)
                ),
            ),
        category = "LoMa",
        name=name
    )
    for name, variant in [("B",'LoMaB'), ("B128",'LoMaB128'), ("L",'LoMaL'), ("G",'LoMaG'), ("R",'LoMaR')]
]


# So bad it messes up the plot scaling
loftr = GradableLocalizer(
    creator=PnPLocalizer.get_creation_function(
        cam2_intrinsic_mtx=labeled_headset_data.intrinsic_cam_mtx,
        extract_and_match_wrapper_config=ExtractAndMatchWrapperConfig(
            extract_and_match=ExtractAndMatchEffLoFTR(matching_threshhold=0.7)
        ),
    ),
    name="E-LoFTR"
)


to_grade_predictors = lightglue_variants+loma_variants+[loftr]
different_matcher_grades = NPredictors1DatasetGrader(
    gradable_pose_predictors=to_grade_predictors,
    headset_data = labeled_headset_data,
    robot_env = robot_env,
)

In [ ]:
import matplotlib.pyplot as plt
from headset_localization import SingleValueErrorType

different_matcher_grades.print_summary()

fig, axes1 = plt.subplots(1,2, figsize = (12, 6))

different_matcher_grades.plot_hz_vs_error(ax=axes1[0], error_type=SingleValueErrorType.MED_TRANSLATIONAL, plot_frontier=False, invert_y=False, plot_legend = False, plot_names=True)
different_matcher_grades.plot_hz_vs_error(ax=axes1[1], error_type=SingleValueErrorType.MED_ROTATIONAL, plot_frontier=False, invert_y = False, plot_names = True)

fig, ax = plt.subplots(1,1, figsize = (12, 6))
different_matcher_grades.plot_hz_vs_error(ax=ax, error_type=SingleValueErrorType.AVG_GRIPPING_ERROR, plot_frontier=False, invert_y=False)
fig, ax = plt.subplots(1,1, figsize = (12, 3.5))
different_matcher_grades.plot_hz_vs_error(ax=ax, error_type=SingleValueErrorType.AVG_NUMBER_POINTS_INLIERS, plot_frontier=False, invert_y=False, plot_legend = False)


## Augmentations
### Rotation Augmentations

In [ ]:
from headset_localization import Augmentation, Rotate180Deg

rot_augmentation_options = [[Augmentation], [Rotate180Deg]]#, [Augmentation, Rotate180Deg]]


super_points = [
    GradableLocalizer(
            creator=PnPLocalizer.get_creation_function(
                cam2_intrinsic_mtx=labeled_headset_data.intrinsic_cam_mtx,
                extract_and_match_wrapper_config=ExtractAndMatchWrapperConfig(
                extract_and_match=ExtractAndLightGlue(extractor="SuperPoint"),
                rotation_augmentations=augs
            ),
        ),
        name=f"SuperPoint + LightGlue, {[str(aug()) for aug in augs]}"
    )
    for augs in rot_augmentation_options
]

sifts = [
    GradableLocalizer(
            creator=PnPLocalizer.get_creation_function(
                cam2_intrinsic_mtx=labeled_headset_data.intrinsic_cam_mtx,
                extract_and_match_wrapper_config=ExtractAndMatchWrapperConfig(
                extract_and_match=ExtractAndLightGlue(extractor="SIFT"),
                rotation_augmentations=augs
            ),
        ),
        name=f"SIFT + LightGlue, {[str(aug()) for aug in augs]}"
    )
    for augs in rot_augmentation_options
]

lomas = [
    GradableLocalizer(
        creator=PnPLocalizer.get_creation_function(
                cam2_intrinsic_mtx=labeled_headset_data.intrinsic_cam_mtx,
                extract_and_match_wrapper_config=ExtractAndMatchWrapperConfig(
                    extract_and_match=ExtractAndMatchLoMa(loma_variant="LoMaB128"),
                    rotation_augmentations=augs
                ),
            ),
        name=f"LoMaB128, {[str(aug()) for aug in augs]}"
    )
    for augs in rot_augmentation_options
]


different_rot_augmentations_predictors = super_points + sifts + lomas
different_rot_aug_grades = NPredictors1DatasetGrader(
    gradable_pose_predictors=different_rot_augmentations_predictors,
    headset_data = labeled_headset_data,
    robot_env = robot_env,
)

In [ ]:
import matplotlib.pyplot as plt

different_rot_aug_grades.print_summary()

fig, ax = plt.subplots(1,1, figsize = (12, 2.5))
different_rot_aug_grades.plot_error_vs_error(ax=ax, error_type_1=SingleValueErrorType.MED_TRANSLATIONAL, error_type_2=SingleValueErrorType.MED_ROTATIONAL, use_in_plot_text=False)


### Crop Augmentations

In [ ]:
lomas_crop = [
    GradableLocalizer(
        creator=PnPLocalizer.get_creation_function(
                cam2_intrinsic_mtx=labeled_headset_data.intrinsic_cam_mtx,
                extract_and_match_wrapper_config=ExtractAndMatchWrapperConfig(
                    extract_and_match=ExtractAndMatchLoMa(loma_variant="LoMaB128"),
                    rotation_augmentations=[Rotate180Deg],
                    crop_augmentations=[amount] if amount is not None else None
                ),
            ),
        category = "LoMaB128",
        name=f"{amount}"
    )
    for amount in [None, 0.3, 0.4, 0.5, 0.6]
]

super_points_crop = [
    GradableLocalizer(
            creator=PnPLocalizer.get_creation_function(
                cam2_intrinsic_mtx=labeled_headset_data.intrinsic_cam_mtx,
                extract_and_match_wrapper_config=ExtractAndMatchWrapperConfig(
                extract_and_match=ExtractAndLightGlue(extractor="SuperPoint"),
                rotation_augmentations=[Rotate180Deg],
                crop_augmentations=[amount] if amount is not None else None
            ),
        ),
        category = "LightGlue",
        name=f"{amount}"
    )
    for amount in [None, 0.3, 0.4, 0.5, 0.6]
]

different_crop_augmentations_predictors = lomas_crop + super_points_crop
different_crop_aug_grades = NPredictors1DatasetGrader(
    gradable_pose_predictors=different_crop_augmentations_predictors,
    headset_data = labeled_headset_data,
    robot_env = robot_env,
)

In [ ]:
import matplotlib.pyplot as plt

#different_crop_aug_grades.print_summary()

#fig, ax = plt.subplots(1,1, figsize = (16, 12))
#different_crop_aug_grades.plot_hz_vs_error(ax=ax, error_type=SingleValueErrorType.AVG_NUMBER_POINTS_INLIERS, plot_frontier=False, use_category=True, invert_y=False)

fig, ax = plt.subplots(1,1, figsize = (12, 2.5))
different_crop_aug_grades.plot_hz_vs_error(ax=ax, error_type=SingleValueErrorType.AVG_TRANSLATIONAL, plot_frontier=False, use_category=True, invert_y=False)
fig, ax = plt.subplots(1,1, figsize = (12, 2.5))
different_crop_aug_grades.plot_hz_vs_error(ax=ax, error_type=SingleValueErrorType.AVG_ROTATIONAL, plot_frontier=False, use_category=True, invert_y=False)

## RANSAC - Parameters
### Different Solvers

In [ ]:
from headset_localization import RansacPoseEstimationConfig, OpenCVPnPSolvers
lg_sp = ExtractAndLightGlue(extractor="SuperPoint")

different_solvers = [
    GradableLocalizer(
            creator=PnPLocalizer.get_creation_function(
                cam2_intrinsic_mtx=labeled_headset_data.intrinsic_cam_mtx,
                extract_and_match_wrapper_config=ExtractAndMatchWrapperConfig(
                extract_and_match=lg_sp,
                crop_augmentations = [0.4],
                ransac_config = RansacPoseEstimationConfig(method=solver),
            ),
        ),
        category = "Var Solvers:",
        name=f"{solver}"
    )
    for solver in [OpenCVPnPSolvers.SOLVEPNP_EPNP, OpenCVPnPSolvers.SOLVEPNP_ITERATIVE, OpenCVPnPSolvers.SOLVEPNP_IPPE] 
]

different_solvers_grades = NPredictors1DatasetGrader(
    gradable_pose_predictors=different_solvers,
    headset_data = labeled_headset_data,
    robot_env = robot_env,
    use_tqdm_for_predictors=True,
    use_tqdm_for_frames=False
)

fig, axes1 = plt.subplots(2,1, figsize = (16, 16))
different_solvers_grades.plot_error_vs_error(
    ax=axes1[0], 
    error_type_1=SingleValueErrorType.AVG_TRANSLATIONAL,
    error_type_2=SingleValueErrorType.AVG_ROTATIONAL,
    plot_frontier=False, use_category=False
)
different_solvers_grades.plot_error_vs_error(
    ax=axes1[1], 
    error_type_1=SingleValueErrorType.AVG_NUMBER_POINTS_INLIERS,
    error_type_2=SingleValueErrorType.AVG_GRIPPING_ERROR,
    invert_y=True
)

cv2.SOLVEPNP_EPNP, cv2.SOLVEPNP_ITERATIVE

### Different numbers of itterations

In [ ]:


different_solvers = [
    GradableLocalizer(
            creator=PnPLocalizer.get_creation_function(
                cam2_intrinsic_mtx=labeled_headset_data.intrinsic_cam_mtx,
                extract_and_match_wrapper_config=ExtractAndMatchWrapperConfig(
                extract_and_match=lg_sp,
                crop_augmentations = [0.4],
                rotation_augmentations=[Rotate180Deg],
                ransac_config = RansacPoseEstimationConfig(iterations = iterations),
            ),
        ),
        category = "Var iterations:",
        name=f"{iterations}"
    )
    for iterations in range(100, 250, 25) 
]

different_solvers_grades = NPredictors1DatasetGrader(
    gradable_pose_predictors=different_solvers,
    headset_data = labeled_headset_data,
    robot_env = robot_env,
    use_tqdm_for_predictors=True,
    use_tqdm_for_frames=False
)

fig, axes1 = plt.subplots(2,1, figsize = (16, 16))
different_solvers_grades.plot_error_vs_error(
    ax=axes1[0], 
    error_type_1=SingleValueErrorType.AVG_TRANSLATIONAL,
    error_type_2=SingleValueErrorType.AVG_ROTATIONAL,
    plot_frontier=False, use_category=False
)
different_solvers_grades.plot_error_vs_error(
    ax=axes1[1], 
    error_type_1=SingleValueErrorType.AVG_NUMBER_POINTS_INLIERS,
    error_type_2=SingleValueErrorType.AVG_GRIPPING_ERROR,
    invert_y=True
)

### Different number of inlier thresholds

In [ ]:
different_inlier_counts = [
    GradableLocalizer(
            creator=PnPLocalizer.get_creation_function(
                cam2_intrinsic_mtx=labeled_headset_data.intrinsic_cam_mtx,
                extract_and_match_wrapper_config=ExtractAndMatchWrapperConfig(
                extract_and_match=lg_sp,
                crop_augmentations = [0.4],
                rotation_augmentations=[Rotate180Deg],
                ransac_config = RansacPoseEstimationConfig(iterations = 300, min_number_inlier_afterwards = min_number_inliers_afterwards),
            ),
        ),
        category = "min number inliers",
        name=f"{min_number_inliers_afterwards}"
    )
    for min_number_inliers_afterwards in range(30, 120, 20) 
]

different_ransac_inlier_count_grades = NPredictors1DatasetGrader(
    gradable_pose_predictors=different_inlier_counts,
    headset_data = labeled_headset_data,
    robot_env = robot_env,
    use_tqdm_for_predictors=True,
    use_tqdm_for_frames=False
)

fig, axes2 = plt.subplots(2,1, figsize = (12, 8))
different_ransac_inlier_count_grades.plot_error_vs_error(
    ax=axes2[0], 
    error_type_1=SingleValueErrorType.AVG_TRANSLATIONAL,
    error_type_2=SingleValueErrorType.AVG_ROTATIONAL,
    plot_frontier=False, use_category=False
)

different_ransac_inlier_count_grades.plot_error_vs_error(
    ax=axes2[1], 
    error_type_1=SingleValueErrorType.SUCCESS_RATE,
    error_type_2=SingleValueErrorType.AVG_GRIPPING_ERROR,
    invert_y=True
)

### Different max reprojection errors

In [ ]:
different_reprojection_error_s = [
    GradableLocalizer(
            creator=PnPLocalizer.get_creation_function(
                cam2_intrinsic_mtx=labeled_headset_data.intrinsic_cam_mtx,
                extract_and_match_wrapper_config=ExtractAndMatchWrapperConfig(
                extract_and_match=lg_sp,
                crop_augmentations = [0.4],
                rotation_augmentations=[Rotate180Deg],
                ransac_config = RansacPoseEstimationConfig(iterations = 500, min_number_inlier_afterwards = 40, reprojection_error=reprojection_error),
            ),
        ),
        category = "Var reproj error",
        name=f"{reprojection_error}"
    )
    for reprojection_error in range(0, 10, 1) 
]

different_ransac_reproj_error_grades = NPredictors1DatasetGrader(
    gradable_pose_predictors=different_reprojection_error_s,
    headset_data = labeled_headset_data,
    robot_env = robot_env,
    use_tqdm_for_predictors=True,
    use_tqdm_for_frames=False
)

fig, axes1 = plt.subplots(2,1, figsize = (12, 8))
different_ransac_reproj_error_grades.plot_error_vs_error(
    ax=axes1[0], 
    error_type_1=SingleValueErrorType.AVG_TRANSLATIONAL,
    error_type_2=SingleValueErrorType.AVG_ROTATIONAL,
    plot_frontier=False, use_category=False
)

different_ransac_reproj_error_grades.plot_error_vs_error(
    ax=axes1[1], 
    error_type_1=SingleValueErrorType.SUCCESS_RATE,
    error_type_2=SingleValueErrorType.AVG_GRIPPING_ERROR,
    invert_y=True
)

### Different confidence threshholds

In [ ]:
different_confidences = [
    GradableLocalizer(
            creator=PnPLocalizer.get_creation_function(
                cam2_intrinsic_mtx=labeled_headset_data.intrinsic_cam_mtx,
                extract_and_match_wrapper_config=ExtractAndMatchWrapperConfig(
                extract_and_match=lg_sp,
                crop_augmentations = [0.4],
                rotation_augmentations=[Rotate180Deg],
                ransac_config = RansacPoseEstimationConfig(iterations = 500, min_number_inlier_afterwards = 40, reprojection_error=5, confidence=conf),
            ),
        ),
        category = "Var confidence error",
        name=f"{conf:.3f}"
    )
    for conf in np.arange(0.8, 0.99, 0.02) 
]

different_ransac_confidences_grades = NPredictors1DatasetGrader(
    gradable_pose_predictors=different_confidences,
    headset_data = labeled_headset_data,
    robot_env = robot_env,
    use_tqdm_for_predictors=True,
    use_tqdm_for_frames=False
)

fig, ax = plt.subplots(1,1, figsize = (12, 8))
different_ransac_confidences_grades.plot_error_vs_error(
    ax=ax, 
    error_type_1=SingleValueErrorType.AVG_TRANSLATIONAL,
    error_type_2=SingleValueErrorType.AVG_ROTATIONAL,
    plot_frontier=False, use_category=False, invert_x=True, invert_y=True
)

## Schedulers
Schedulers will switch through the robot images if a pose cant be predicted

In [ ]:
from headset_localization import Scheduler, EMAScheduler, BlockingEMAScheduler, RansacPoseEstimationConfig

super_points_shedulers = [
    GradableLocalizer(
            creator=PnPLocalizer.get_creation_function(
                cam2_intrinsic_mtx=labeled_headset_data.intrinsic_cam_mtx,
                extract_and_match_wrapper_config=ExtractAndMatchWrapperConfig(
                extract_and_match=ExtractAndLightGlue(extractor="SuperPoint"),
                rotation_augmentations=[Rotate180Deg],
                ransac_config = RansacPoseEstimationConfig(min_number_inlier_afterwards = 30, reprojection_error=4),
                scheduler=scheduler
            ),
        ),
        name=f"{scheduler(1)}"
    )
    for scheduler in [Scheduler, EMAScheduler, BlockingEMAScheduler]
]


different_scheduler_aug_grades = NPredictors1DatasetGrader(
    gradable_pose_predictors=super_points_shedulers,
    headset_data = labeled_headset_data,
    robot_env = robot_env,
)

In [ ]:
from headset_localization import TimeSeriesErrorType

import matplotlib.pyplot as plt

different_scheduler_aug_grades.print_summary()

fig, axes = plt.subplots(3,1, figsize = (12, 12))
different_scheduler_aug_grades.plot_time_series_error(ax=axes[0], error_type=TimeSeriesErrorType.ABS_ROTATIONAL)
different_scheduler_aug_grades.plot_time_series_error(ax=axes[1], error_type=TimeSeriesErrorType.ABS_TRANSLATIONAL)
different_scheduler_aug_grades.plot_error_vs_error(ax=axes[2], error_type_1=SingleValueErrorType.SUCCESS_RATE, error_type_2=SingleValueErrorType.AVG_GRIPPING_ERROR, invert_y=True)


